# Native 2-Channel + Real Metadata, Leakage-Free Split

Completes the channel x metadata 2x2 on the **leakage-free** spatial-block
split. This cell tests real VV/VH channels **and** real metadata together.

**Why this run exists.** The original 2x2 (`pcrtc/04` and its siblings) was
evaluated on the naive random split, which leaked: 100% of patches overlap a
neighbour, so validation patches shared ground with training patches. Two of
the four cells have since been retrained honestly -- `pcrtc/10` (baseline,
ZNCC 0.1867) and `pcrtc/09` (metadata, ZNCC 0.2344). This notebook and its
sibling fill in the two that were never redone, so the ablation can be
reported as a result rather than as pre-correction model selection.

**Leaky value for this cell was 0.219.** Expect it to fall
substantially -- the correction cut the metadata cell from 0.519 to 0.2344,
roughly a fivefold reduction in apparent effect size.

**Read the outcome against the noise floor, not against zero.** Two runs
differing only in initialisation land ~0.05 ZNCC apart. On the honest split
the metadata advantage over baseline is +0.048 -- already at that floor. The
channel effect in the leaky data was only -0.04, i.e. smaller still, so an
inconclusive result here is a likely and perfectly reportable outcome. Run
`dem_unet/06` first so the threshold is measured rather than assumed.

**What changes from `pcrtc/09`:**

| | value |
|---|---|
| channels per view | 2 (native VV/VH, **no repetition**) |
| `cond_channels_per_view` | 2 -- a workstation patch to `ConditionalUNet` |
| metadata | real acquisition attrs |
| `SPLIT_SEED` | 42, fixed -- identical validation patches |
| `INIT_SEED` | 42, matching `09` |
| everything else | identical |

**This notebook does not execute automatically. Run cells top to bottom.**

## GPU configuration

In [1]:
import os
import sys
import json
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio

assert torch.cuda.is_available(), 'CUDA is required. Run this notebook on the GPU environment.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 4090


## Configuration

Every value below matches `pcrtc/09` except `INIT_SEED`. Change only
`INIT_SEED` if you want to run a third replicate later.

In [2]:
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = WORKING_REPO / 'tessa_baseline'
REGION = 'tuk'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_tuk_pcrtc'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_SEED = 42     # MUST stay 42 -- identical split to 09
INIT_SEED = 42      # matches 09; the conditioning setup is what varies

CONTEXT_K = 3
TARGET_HW = (256, 256)
BATCH_SIZE = 8
EPOCHS = 100
TIMESTEPS = 1000
LEARNING_RATE = 1e-4
VAL_FRACTION = 0.15
NOISE_SCHEDULE = 'linear'
ATTENTION_VARIANT = 'default'
LIDAR_SURVEY_DATE = __import__('datetime').date(2024, 4, 16)
BLOCK_SIZE_M = 1024.0
BUFFER_M = 150.0

CHECKPOINT_NAME = f's1_{REGION}_pcrtc_native2ch_realattrs_spatialsplit_unet_best.pth'
METRICS_FILENAME = 's1_pcrtc_native2ch_realattrs_spatialsplit_validation_metrics.json'
THIS_LABEL = 'both (2ch, real)'
BASELINE_METRICS = 's1_pcrtc_realattrs_spatialsplit_validation_metrics.json'      # 09
DEM_METRICS = 's1_pcrtc_dem_realattrs_spatialsplit_validation_metrics.json'       # 03 (inert DEM)

print('Cell under test: both (2ch, real)')
print(f'SPLIT_SEED={SPLIT_SEED}, INIT_SEED={INIT_SEED}, CONTEXT_K={CONTEXT_K}')
print('Checkpoint:', CHECKPOINT_DIR / CHECKPOINT_NAME)

Cell under test: both (2ch, real)
SPLIT_SEED=42, INIT_SEED=42, CONTEXT_K=3
Checkpoint: /cs/student/project_msc/2025/aibh/jiayiche/checkpoints/s1_tuk_pcrtc_native2ch_realattrs_spatialsplit_unet_best.pth


## Imports and seeding

In [3]:
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddim
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(INIT_SEED)   # NOT SPLIT_SEED -- this is the variable under test
print('Global seed set to', INIT_SEED)

Global seed set to 42


## Dataset adapter -- unchanged from `09`

In [4]:
def build_real_attrs(s1_path, times, context_k):
    attrs_path = s1_path / 'attrs.json'
    attrs_list = json.load(open(attrs_path)) if attrs_path.exists() else []
    vecs = []
    for time_path in times:
        idx = int(time_path.stem[1:])
        a = attrs_list[idx] if idx < len(attrs_list) else {}
        if a.get('acquisition_date'):
            import datetime as dt
            acq_date = dt.date.fromisoformat(a['acquisition_date'])
            age_norm = (acq_date - LIDAR_SURVEY_DATE).days / 30.0
        else:
            age_norm = 0.0
        orbit_dir = 1.0 if a.get('orbit_direction') == 'ASCENDING' else 0.0
        rel_orbit = (a.get('relative_orbit_number') or 0) / 175.0
        vecs.append([age_norm, orbit_dir, rel_orbit, 0.0, 0.0, 0.0, 0.0, 0.0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()


class LidarS1Dataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, patch_ids, context_k=3, target_hw=(256, 256)):
        self.s1_dir = Path(s1_dir)
        self.lidar_dir = Path(lidar_dir)
        self.patch_ids = list(patch_ids)
        self.context_k = context_k
        self.target_hw = target_hw

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, index):
        patch_id = self.patch_ids[index]
        with rasterio.open(self.lidar_dir / f'lidar_patch_{patch_id}.tif') as src:
            raw = src.read().astype(np.float32)
        target = raw[0]
        mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        valid_count = max(1, int(mask.sum()))
        patch_mean = float(target[mask].sum() / valid_count)
        target = (target - patch_mean) * mask

        s1_path = self.s1_dir / f's1_patch_{patch_id}'
        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k:
            raise RuntimeError(f'{s1_path} has fewer than {self.context_k} Sentinel-1 times')
        views = []
        for time_path in times:
            with rasterio.open(time_path) as src:
                sar = src.read()[:2].astype(np.float32)
            sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
            sar = np.maximum(sar, 1e-12)
            sar = 10.0 * np.log10(sar)
            sar_tensor = torch.from_numpy(sar).unsqueeze(0)
            sar_tensor = F.interpolate(sar_tensor, size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)
            views.append(sar_tensor)
        condition = torch.cat(views, dim=0)
        attrs = build_real_attrs(s1_path, times, self.context_k)
        return {'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask),
                's1': condition.float(), 'attrs': attrs,
                'patch_mean': torch.tensor(patch_mean), 'patch_id': patch_id}

## Spatial-block split -- `SPLIT_SEED`, not `INIT_SEED`

Copied verbatim from `09`. The only edit is `random.Random(SPLIT_SEED)`
in place of `random.Random(SEED)`, which keeps the split pinned to 42
while the initialisation seed varies. The assertion below is the guard
that makes this whole experiment valid.

In [5]:
lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
paired_ids = sorted(lidar_ids & s1_ids)
assert paired_ids, 'No paired Sentinel-1/LiDAR patches found.'

def patch_centroid(patch_id):
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{patch_id}.tif') as src:
        b = src.bounds
    return ((b.left + b.right) / 2.0, (b.bottom + b.top) / 2.0)

centroids = {pid: patch_centroid(pid) for pid in paired_ids}

def block_id_and_boundary_distance(cx, cy, block_size):
    bx, by = int(cx // block_size), int(cy // block_size)
    dx = min(cx - bx * block_size, (bx + 1) * block_size - cx)
    dy = min(cy - by * block_size, (by + 1) * block_size - cy)
    return (bx, by), min(dx, dy)

blocks = {}
dropped_buffer = []
for pid, (cx, cy) in centroids.items():
    bid, boundary_dist = block_id_and_boundary_distance(cx, cy, BLOCK_SIZE_M)
    if boundary_dist < BUFFER_M:
        dropped_buffer.append(pid)
        continue
    blocks.setdefault(bid, []).append(pid)

block_ids = list(blocks.keys())
random.Random(SPLIT_SEED).shuffle(block_ids)   # pinned to 42, independent of INIT_SEED

target_val_patches = int(len(paired_ids) * VAL_FRACTION)
val_ids, train_ids = [], []
running_val_count = 0
for bid in block_ids:
    if running_val_count < target_val_patches:
        val_ids.extend(blocks[bid])
        running_val_count += len(blocks[bid])
    else:
        train_ids.extend(blocks[bid])

print(f'Split: train={len(train_ids)}, val={len(val_ids)}, dropped={len(dropped_buffer)}')
assert (len(train_ids), len(val_ids)) == (534, 255), (
    f'Split is {len(train_ids)}/{len(val_ids)}, expected 534/255. The split has drifted '
    f'from 09 and no comparison below would be valid. Stop and investigate.')
print('Split confirmed identical to 09.')

train_dataset = LidarS1Dataset(S1_DIR, LIDAR_DIR, train_ids, CONTEXT_K, TARGET_HW)
val_dataset = LidarS1Dataset(S1_DIR, LIDAR_DIR, val_ids, CONTEXT_K, TARGET_HW)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

Split: train=534, val=255, dropped=887
Split confirmed identical to 09.


## Model, scheduler, optimizer

In [6]:
# cond_channels_per_view=2 is a patch applied to the workstation's tessa_baseline
# (see CONCEPTS.md ~line 634). It is NOT in the upstream ConditionalUNet, so this
# fails loudly rather than silently training a differently-shaped model.
try:
    model = ConditionalUNet(
        in_channels=1, cond_channels=4 * CONTEXT_K, attr_dim=8 * CONTEXT_K,
        base_channels=128, embed_dim=256, unet_depth=4,
        attention_variant=ATTENTION_VARIANT, cond_k=CONTEXT_K,
        cond_channels_per_view=2,
    ).to(DEVICE)
except TypeError as exc:
    raise RuntimeError(
        'ConditionalUNet does not accept cond_channels_per_view. This notebook needs '
        'the native-2-channel patch that pcrtc/04 and pcrtc/05 relied on. Check that '
        'TESSA_REPO points at the patched workstation copy, not an unpatched clone.'
    ) from exc

scheduler = LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else CosineDiffusionScheduler(TIMESTEPS, device=DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print('Trainable parameters:', f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'Conditioning: {CONTEXT_K} views x 2 native channels (no repetition)')

Trainable parameters: 104,624,731
Conditioning: 3 views x 2 native channels (no repetition)


## Training loop -- identical to `09`

`09`'s best validation loss arrived at **epoch 93 of 100**, with validation
wandering in the 0.0137-0.0174 band throughout and rising streaks of up to
3 epochs. Early validation movement carries no signal here. Do not stop
this run early.

In [7]:
from torch.cuda.amp import autocast, GradScaler

def masked_mse(prediction, target, mask):
    valid = mask.bool().unsqueeze(1)
    error = (prediction - target) ** 2
    return error[valid].mean()

scaler = GradScaler()
history = {'train_loss': [], 'val_loss': []}
best_val = float('inf')
best_epoch = -1
for epoch in range(EPOCHS):
    model.train()
    train_total = 0.0
    for batch in train_loader:
        target = batch['lidar'].to(DEVICE, non_blocking=True)
        condition = batch['s1'].to(DEVICE, non_blocking=True)
        attrs = batch['attrs'].to(DEVICE, non_blocking=True)
        mask = batch['mask'].to(DEVICE, non_blocking=True)
        timestep = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            noisy = scheduler.q_sample(target, timestep)
            prediction = model(noisy, condition, attrs, timestep)
            loss = masked_mse(prediction, target, mask)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_total += loss.item()
    model.eval()
    val_total = 0.0
    with torch.no_grad():
        for batch in val_loader:
            target = batch['lidar'].to(DEVICE, non_blocking=True)
            condition = batch['s1'].to(DEVICE, non_blocking=True)
            attrs = batch['attrs'].to(DEVICE, non_blocking=True)
            mask = batch['mask'].to(DEVICE, non_blocking=True)
            timestep = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
            with autocast():
                prediction = model(scheduler.q_sample(target, timestep), condition, attrs, timestep)
                val_total += masked_mse(prediction, target, mask).item()
    train_loss = train_total / max(1, len(train_loader))
    val_loss = val_total / max(1, len(val_loader))
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    print(f'Epoch {epoch + 1:03d}/{EPOCHS}: train={train_loss:.6f} val={val_loss:.6f}')
    if val_loss < best_val:
        best_val = val_loss
        best_epoch = epoch + 1
        torch.save({'model_state_dict': model.state_dict(),
                    'config': {'context_k': CONTEXT_K, 'timesteps': TIMESTEPS,
                               'noise_schedule': NOISE_SCHEDULE, 'region': REGION,
                               'split_seed': SPLIT_SEED, 'init_seed': INIT_SEED},
                    'epoch': best_epoch, 'val_loss': val_loss},
                   CHECKPOINT_DIR / CHECKPOINT_NAME)

print(f'\nBest val {best_val:.6f} at epoch {best_epoch}  (09 reached 0.013706 at epoch 93)')

/tmp/ipykernel_215887/1107549189.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_215887/1107549189.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_215887/1107549189.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 001/100: train=0.060165 val=0.026373
Epoch 002/100: train=0.018653 val=0.018902
Epoch 003/100: train=0.016006 val=0.017390
Epoch 004/100: train=0.014930 val=0.016938
Epoch 005/100: train=0.015019 val=0.017643
Epoch 006/100: train=0.014853 val=0.016793
Epoch 007/100: train=0.014676 val=0.015825
Epoch 008/100: train=0.014595 val=0.016328
Epoch 009/100: train=0.014098 val=0.016909
Epoch 010/100: train=0.013543 val=0.015349
Epoch 011/100: train=0.014699 val=0.014932
Epoch 012/100: train=0.013862 val=0.014909
Epoch 013/100: train=0.014071 val=0.015428
Epoch 014/100: train=0.013504 val=0.015501
Epoch 015/100: train=0.013125 val=0.016735
Epoch 016/100: train=0.014493 val=0.016237
Epoch 017/100: train=0.014040 val=0.016060
Epoch 018/100: train=0.014370 val=0.015850
Epoch 019/100: train=0.013654 val=0.015630
Epoch 020/100: train=0.012989 val=0.016396
Epoch 021/100: train=0.013663 val=0.015236
Epoch 022/100: train=0.013805 val=0.015124
Epoch 023/100: train=0.014363 val=0.015377
Epoch 024/1

## Evaluation -- identical metric suite and sampler

In [8]:
checkpoint = torch.load(CHECKPOINT_DIR / CHECKPOINT_NAME, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
sampler = p_sample_loop_ddim
metric_rows = []
with torch.no_grad():
    for batch in val_loader:
        target = batch['lidar'].to(DEVICE)
        condition = batch['s1'].to(DEVICE)
        attrs = batch['attrs'].to(DEVICE)
        mask = batch['mask'].to(DEVICE).bool()
        prediction = sampler(model, scheduler, target.shape, condition, attrs, DEVICE)
        means = batch['patch_mean'].to(DEVICE).view(-1, 1, 1, 1)
        gt_absolute = target + means
        pred_absolute = prediction + means
        for i, patch_id in enumerate(batch['patch_id']):
            gt_i, pred_i, mask_i = gt_absolute[i], pred_absolute[i], mask[i]
            gt_valid = gt_i.squeeze()[mask_i].cpu().numpy()
            pred_valid = pred_i.squeeze()[mask_i].cpu().numpy()
            metric_rows.append({
                'patch_id': patch_id,
                'rmse_m': float(rmse(gt_i, pred_i, mask_i).item()),
                'bias_m': float(bias(gt_i, pred_i, mask_i).item()),
                'sigma_error_pct': float(sigma_error(gt_i, pred_i, mask_i).item()),
                'normal_angle_error_deg': float(normal_angle_error(gt_i, pred_i, mask_i, pixel_size=1.0, degrees=True).item()),
                'jsd': float(average_jsd_multiscale(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'psd_rmse': float(log_psd_rmse(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'zncc': float(zncc(gt_i, pred_i, mask_i).item()),
                'gt_std_val': float(gt_valid.std()) if gt_valid.size > 0 else float('nan'),
                'pred_std_val': float(pred_valid.std()) if pred_valid.size > 0 else float('nan'),
            })

metrics_path = OUTPUT_DIR / METRICS_FILENAME
with metrics_path.open('w') as h:
    json.dump(metric_rows, h, indent=2)
print('Saved:', metrics_path)
print('Mean metrics:', {k: float(np.nanmean([r[k] for r in metric_rows])) for k in metric_rows[0] if k != 'patch_id'})

Saved: /cs/student/project_msc/2025/aibh/jiayiche/s1_training_outputs/s1_pcrtc_native2ch_realattrs_spatialsplit_validation_metrics.json
Mean metrics: {'rmse_m': 0.2087922500044692, 'bias_m': 0.008628509949673624, 'sigma_error_pct': 20.642488207127535, 'normal_angle_error_deg': 2.0435773690541583, 'jsd': 0.09706686546332112, 'psd_rmse': 1.3035040070028867, 'zncc': 0.2012214774693883, 'gt_std_val': 0.17209622516643766, 'pred_std_val': 0.15835586949890734}


## The 2x2, all four cells on the honest split

Every cell below used the identical spatial-block split, so the comparison is
internally valid. The leaky numbers are printed alongside only to show how far
the correction moved each one -- they are not comparable to these.

The final block checks each gap against the measured run-to-run floor. A gap
smaller than the floor is not an effect, however tidy it looks.

In [10]:
def load_rows(name):
    p = OUTPUT_DIR / name
    if not p.exists():
        print(f'  (missing: {name})')
        return None
    return {r['patch_id']: r for r in json.load(p.open())}

# The 2x2, all four cells on the SAME leakage-free split
cells = {
    'baseline (repeat, zeros)': 's1_pcrtc_baseline_spatialsplit_validation_metrics.json',
    'metadata (repeat, real)': 's1_pcrtc_realattrs_spatialsplit_validation_metrics.json',
    'channels (2ch, zeros)': 's1_pcrtc_native2ch_spatialsplit_validation_metrics.json',
    'both (2ch, real)': 's1_pcrtc_native2ch_realattrs_spatialsplit_validation_metrics.json',
}
runs = {k: load_rows(v) for k, v in cells.items()}
runs[THIS_LABEL] = {r['patch_id']: r for r in metric_rows}
runs = {k: v for k, v in runs.items() if v}

common = set.intersection(*(set(v) for v in runs.values()))
print(f'\ncells available: {len(runs)}   patches common to all: {len(common)}\n')

METRICS = ['zncc', 'rmse_m', 'sigma_error_pct', 'jsd', 'psd_rmse', 'pred_std_val']
w = max(24, max(len(k) for k in runs) + 2)
print(f"{'metric':<18}" + ''.join(f'{k:>{w}}' for k in runs))
for m in METRICS:
    print(f'{m:<18}' + ''.join(f'{np.nanmean([r[i][m] for i in common]):>{w}.4f}' for r in runs.values()))

print('\nLeaky reference (naive split, NOT comparable -- shown only to see how much')
print('the correction moved each cell):  baseline 0.286 | channels 0.245 | '
      'metadata 0.519 | both 0.219')

this = {r['patch_id']: r for r in metric_rows}

# The k=3 configuration has THREE runs. Comparing a new configuration against any
# single one of them (especially 09, which is the lowest) overstates the effect.
# Compare against the distribution instead.
# this run's rows, keyed by patch id (defined here so the block is self-contained)
this = {r['patch_id']: r for r in metric_rows}

K3_RUNS = {
    '09 (seed 42)': 's1_pcrtc_realattrs_spatialsplit_validation_metrics.json',
    'seed 43': 's1_pcrtc_realattrs_spatialsplit_seed43_validation_metrics.json',
    'inert-DEM (= 3rd seed)': 's1_pcrtc_dem_realattrs_spatialsplit_validation_metrics.json',
}
k3 = {k: load_rows(v) for k, v in K3_RUNS.items()}
k3 = {k: v for k, v in k3.items() if v}

if len(k3) >= 2:
    ids = sorted(set.intersection(*(set(v) for v in k3.values())) & set(common))
    per_run = {k: float(np.nanmean([v[i]['zncc'] for i in ids])) for k, v in k3.items()}
    vals = np.array(list(per_run.values()))
    mean_k3, sd_k3, rng_k3 = vals.mean(), vals.std(ddof=1) if len(vals) > 1 else np.nan, vals.max() - vals.min()
    this_z = float(np.nanmean([this[i]['zncc'] for i in ids]))

    print(f'\n--- against the k=3 seed DISTRIBUTION ({len(vals)} runs) ---')
    for k, v in per_run.items():
        print(f'  {k:<26} {v:.4f}')
    print(f'  {"mean":<26} {mean_k3:.4f}   s.d. {sd_k3:.4f}   range {rng_k3:.4f}')
    print(f'  {"this run":<26} {this_z:.4f}')

    d_mean = this_z - mean_k3
    n_sd = d_mean / sd_k3 if sd_k3 and np.isfinite(sd_k3) else float('nan')
    print(f'\n  vs seed MEAN : {d_mean:+.4f}  ({n_sd:+.1f} s.d.)')
    print(f'  vs seed range: {"OUTSIDE" if this_z > vals.max() or this_z < vals.min() else "INSIDE"} '
          f'[{vals.min():.4f}, {vals.max():.4f}]')
    if abs(n_sd) >= 2:
        print('  -> exceeds two standard deviations: a real effect')
    elif this_z > vals.max() or this_z < vals.min():
        print('  -> outside the observed range but under 2 s.d.: suggestive, not established')
    else:
        print('  -> INSIDE the range of runs that differ only by seed: NOT established')

    print('\n  Other metrics vs their k=3 seed ranges (these catch effects ZNCC misses):')
    for m in ['rmse_m', 'sigma_error_pct', 'pred_std_val', 'psd_rmse', 'jsd']:
        r = np.array([np.nanmean([v[i][m] for i in ids]) for v in k3.values()])
        t = float(np.nanmean([this[i][m] for i in ids]))
        out = t > r.max() or t < r.min()
        print(f'    {m:<18} k=3 [{r.min():.4f}, {r.max():.4f}]   this {t:.4f}   '
              f'{"OUTSIDE" if out else "inside"}')
else:
    print('\nOnly one k=3 run found. Run dem_unet/06 before interpreting any gap.')


cells available: 4   patches common to all: 255

metric              baseline (repeat, zeros)   metadata (repeat, real)     channels (2ch, zeros)          both (2ch, real)
zncc                                  0.1867                    0.2344                    0.2068                    0.2012
rmse_m                                0.2360                    0.1945                    0.2227                    0.2088
sigma_error_pct                      30.4628                   22.7296                   23.9357                   20.6425
jsd                                   0.0889                    0.1180                    0.0940                    0.0971
psd_rmse                              1.1130                    1.3092                    0.9724                    1.3035
pred_std_val                          0.1971                    0.1384                    0.1798                    0.1584

Leaky reference (naive split, NOT comparable -- shown only to see how much
the correctio